# Risk-Adjusted Portfolio Optimization: LSTM vs. Markowitz MVO
**AIM 5005 Final Project** — Alexy Skoutnev, Shaun Mukahanana, Tadiwanashe Chiremba

This notebook walks through the full pipeline:
1. Data collection & preprocessing
2. Markowitz MVO baseline (efficient frontier, min-variance, max-Sharpe)
3. Rolling backtest
4. Volatility regime analysis
5. Risk metrics evaluation

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# Project modules
from src.config import TICKERS, START_DATE, END_DATE, ROLLING_WINDOW, TRAIN_RATIO, VAL_RATIO
from src.data.fetcher import fetch_prices, fetch_vix
from src.data.preprocessing import compute_log_returns, compute_simple_returns, split_data
from src.data.regime import label_regimes_vix, label_regimes_rolling_std
from src.models.mvo import (
    estimate_parameters, portfolio_performance,
    minimum_variance_weights, max_sharpe_weights,
    efficient_frontier, rolling_backtest
)
from src.risk_metrics import compute_all_metrics, portfolio_turnover
from src.visualization import (
    plot_efficient_frontier, plot_cumulative_returns,
    plot_weight_allocation, plot_regime_returns,
    plot_drawdown, plot_rolling_sharpe
)

print(f"Tickers: {TICKERS}")
print(f"Date range: {START_DATE} to {END_DATE}")

## 1. Data Collection & Preprocessing

In [ ]:
# Fetch adjusted close prices for sector ETFs and VIX
prices = fetch_prices(TICKERS, START_DATE, END_DATE, cache_dir="data/raw")
vix = fetch_vix(START_DATE, END_DATE)

print(f"Price data shape: {prices.shape}")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"\nMissing values per ticker:\n{prices.isna().sum()}")
prices.tail()

In [ ]:
# Compute log returns (for modeling) and simple returns (for portfolio P&L)
log_returns = compute_log_returns(prices)
simple_returns = compute_simple_returns(prices)

print(f"Log returns shape: {log_returns.shape}")
log_returns.describe().round(4)

In [ ]:
# Temporal train/val/test split
train, val, test = split_data(log_returns, TRAIN_RATIO, VAL_RATIO)

print(f"Train: {len(train)} days ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Val:   {len(val)} days ({val.index[0].date()} to {val.index[-1].date()})")
print(f"Test:  {len(test)} days ({test.index[0].date()} to {test.index[-1].date()})")

## 2. Exploratory Data Analysis

In [ ]:
# Normalized price chart (rebased to 1.0)
normalized = prices / prices.iloc[0]

fig, ax = plt.subplots(figsize=(14, 6))
normalized.plot(ax=ax, linewidth=1)
ax.set_title("Normalized Sector ETF Prices (2010-2025)")
ax.set_ylabel("Growth of $1")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of log returns
corr = log_returns.corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdYlGn", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
fig.colorbar(im, ax=ax, label="Correlation")
ax.set_title("Return Correlation Matrix")

# Annotate cells with correlation values
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)

plt.tight_layout()
plt.show()

## 3. Markowitz MVO Baseline

In [ ]:
# Estimate parameters from training data
mean_ret, cov_mat = estimate_parameters(train)

print("Annualized Expected Returns:")
for ticker, ret in zip(TICKERS, mean_ret):
    print(f"  {ticker}: {ret:.4f} ({ret*100:.2f}%)")

print(f"\nCovariance matrix shape: {cov_mat.shape}")

In [ ]:
# Compute optimal portfolios
min_var_w = minimum_variance_weights(cov_mat)
max_sharpe_w = max_sharpe_weights(mean_ret, cov_mat)

min_var_ret, min_var_vol = portfolio_performance(min_var_w, mean_ret, cov_mat)
max_sharpe_ret, max_sharpe_vol = portfolio_performance(max_sharpe_w, mean_ret, cov_mat)

print("=== Minimum Variance Portfolio ===")
print(f"  Return: {min_var_ret:.4f} ({min_var_ret*100:.2f}%)")
print(f"  Volatility: {min_var_vol:.4f} ({min_var_vol*100:.2f}%)")
print(f"  Sharpe: {min_var_ret/min_var_vol:.4f}")
print(f"  Weights:")
for ticker, w in zip(TICKERS, min_var_w):
    if w > 0.01:
        print(f"    {ticker}: {w:.4f} ({w*100:.1f}%)")

print(f"\n=== Maximum Sharpe Portfolio ===")
print(f"  Return: {max_sharpe_ret:.4f} ({max_sharpe_ret*100:.2f}%)")
print(f"  Volatility: {max_sharpe_vol:.4f} ({max_sharpe_vol*100:.2f}%)")
print(f"  Sharpe: {max_sharpe_ret/max_sharpe_vol:.4f}")
print(f"  Weights:")
for ticker, w in zip(TICKERS, max_sharpe_w):
    if w > 0.01:
        print(f"    {ticker}: {w:.4f} ({w*100:.1f}%)")

In [ ]:
# Plot efficient frontier
frontier = efficient_frontier(mean_ret, cov_mat, n_points=100)

fig = plot_efficient_frontier(
    frontier,
    max_sharpe_point=(max_sharpe_vol, max_sharpe_ret),
    min_var_point=(min_var_vol, min_var_ret),
)
plt.show()

## 4. Rolling Backtest

In [ ]:
# Run rolling backtests for both strategies
bt_max_sharpe = rolling_backtest(log_returns, window=ROLLING_WINDOW, strategy="max_sharpe")
bt_min_var = rolling_backtest(log_returns, window=ROLLING_WINDOW, strategy="min_variance")

# Equal-weight benchmark (buy and hold 1/N)
equal_weight_returns = log_returns.mean(axis=1)
# Align to backtest dates
equal_weight_aligned = equal_weight_returns.loc[bt_max_sharpe.index]
bt_equal_weight = pd.DataFrame({
    "portfolio_return": equal_weight_aligned,
    "cumulative_return": equal_weight_aligned.cumsum().apply(np.exp) - 1
}, index=bt_max_sharpe.index)

print(f"Backtest period: {bt_max_sharpe.index[0].date()} to {bt_max_sharpe.index[-1].date()}")
print(f"Out-of-sample days: {len(bt_max_sharpe)}")

In [ ]:
# Cumulative returns comparison
fig = plot_cumulative_returns({
    "Max Sharpe": bt_max_sharpe,
    "Min Variance": bt_min_var,
    "Equal Weight (1/N)": bt_equal_weight,
})
plt.show()

In [ ]:
# Weight allocation over time (Max Sharpe)
fig = plot_weight_allocation(
    bt_max_sharpe, TICKERS,
    title="Max Sharpe — Portfolio Weight Allocation Over Time"
)
plt.show()

In [ ]:
# Drawdown analysis
fig = plot_drawdown(bt_max_sharpe)
plt.show()

In [ ]:
# Rolling Sharpe ratio
fig = plot_rolling_sharpe(bt_max_sharpe, window=63)
plt.show()

## 5. Risk Metrics

In [ ]:
# Compute risk metrics for all strategies
strategies = {
    "Max Sharpe": bt_max_sharpe["portfolio_return"],
    "Min Variance": bt_min_var["portfolio_return"],
    "Equal Weight": bt_equal_weight["portfolio_return"],
}

metrics_list = []
for name, returns_series in strategies.items():
    m = compute_all_metrics(returns_series)
    m["strategy"] = name
    # Add turnover if weight data is available
    if name == "Max Sharpe":
        m["turnover"] = portfolio_turnover(bt_max_sharpe[TICKERS])
    elif name == "Min Variance":
        m["turnover"] = portfolio_turnover(bt_min_var[TICKERS])
    else:
        m["turnover"] = 0.0
    metrics_list.append(m)

metrics_df = pd.DataFrame(metrics_list).set_index("strategy")
metrics_df = metrics_df.round(4)
metrics_df

## 6. Volatility Regime Analysis

In [ ]:
# Label regimes using VIX
vix_aligned = vix.reindex(bt_max_sharpe.index).ffill().bfill()
regimes = label_regimes_vix(vix_aligned)

print("Regime distribution (out-of-sample period):")
print(regimes.value_counts())
print(f"\nVIX stats during backtest:")
print(vix_aligned.describe().round(2))

In [ ]:
# Return distributions by regime
fig = plot_regime_returns(
    bt_max_sharpe["portfolio_return"],
    regimes,
    title="Max Sharpe Portfolio Returns by Volatility Regime"
)
plt.show()

In [ ]:
# Per-regime risk metrics
regime_metrics = []
for regime_label in ["low", "medium", "high"]:
    mask = regimes == regime_label
    if mask.sum() > 10:
        regime_ret = bt_max_sharpe.loc[mask, "portfolio_return"]
        m = compute_all_metrics(regime_ret)
        m["regime"] = regime_label
        m["n_days"] = int(mask.sum())
        regime_metrics.append(m)

regime_df = pd.DataFrame(regime_metrics).set_index("regime")
regime_df = regime_df.round(4)
print("Max Sharpe — Risk Metrics by Volatility Regime:")
regime_df

## 7. VIX Over Time

In [ ]:
# VIX with regime shading
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(vix_aligned.index, vix_aligned, linewidth=0.8, color="#2c3e50")
ax.axhline(y=15, color="green", linestyle="--", alpha=0.5, label="Low/Medium threshold (15)")
ax.axhline(y=25, color="red", linestyle="--", alpha=0.5, label="Medium/High threshold (25)")
ax.fill_between(vix_aligned.index, 0, 15, alpha=0.1, color="green")
ax.fill_between(vix_aligned.index, 25, vix_aligned.max() + 5, alpha=0.1, color="red")
ax.set_title("VIX — Volatility Regimes")
ax.set_ylabel("VIX")
ax.set_xlabel("Date")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Next Steps

- **LSTM model**: Train LSTM on rolling windows to forecast next-period returns
- **LSTM-enhanced portfolio**: Use LSTM forecasts as inputs to MVO or proportional allocation
- **Comparison**: Compare LSTM portfolio vs MVO baseline across all metrics and regimes